# 01 · Suppression-mechanism confirmation  —  `DIFF@pivot = ATTN(q = −PROGRESS, k = TASK)`

Hypothesis: at the pivot the goal heads compute the leftover task by a single attention op whose
**query is a function of PROGRESS** (the done-set) and whose **keys/values are the TASK spec**. The
query *suppresses* attention onto the task items already satisfied by PROGRESS; the surviving mass is
the frontier item, written by the OV.

Battery (one 3-predicate example, 2–3 instances — flow-polishing only, no aggregate stats):
- **A1** graded suppression + OV-write norm — does attention onto item-k collapse exactly when PROGRESS completes k?
- **A2** anti-alignment — is the query update from completing item i anti-parallel to item-i's task keys?
- **A3** sign necessity — reflecting the done-update through baseline should *not* skip to the frontier.
- **B1** query linearity (geometric) — `Δq({a,b}) ≈ Δq({a}) + Δq({b})`?
- **B2** compositional causal injection — inject `q({a})+q({b})−q({})`; does it *generate* item c only? saturation → boundary.

In [ ]:
%load_ext autoreload
%autoreload 2
import sys; sys.path.append("..")            # so `_diff_primitives` is importable
import torch, numpy as np
import _diff_primitives as P
P.setup("Qwen/Qwen2.5-Coder-32B-Instruct")   # loads model + head masks (once)
from _diff_primitives import *                # helper functions into scope
mt = P.mt
GOAL_HEADS, BOUNDARY_HEADS, UNION_HEADS, GROUP = P.GOAL_HEADS, P.BOUNDARY_HEADS, P.UNION_HEADS, P.GROUP

In [ ]:
# ── multipredicate example: 3 INDEPENDENT predicates, one assignment line each ──
# Each item can be implemented in any order (all orderings correct) — the pivot choice is a
# genuine "which sub-task next", not forced by a data dependency.
ITEMS = {
    "a": ("the list of words longer than three characters", "a = [w for w in words if len(w) > 3]"),
    "b": ("the count of palindromic words",                 "b = sum(w == w[::-1] for w in words)"),
    "c": ("the length of the longest word",                 "c = max(len(w) for w in words)"),
}
ORDER = ["a", "b", "c"]
TASK_SPEC = "\n".join(f"({i+1}) `{k}`: {ITEMS[k][0]}" for i, k in enumerate(ORDER))
QUESTION = ("Write a Python program that implements a function `func` which, given a list of "
            "strings `words`, computes:\n" + TASK_SPEC + "\n")

def build_prefill(done, extra=""):
    """Prefill implementing the assignments in `done` (kept in ORDER), pivot at the next indent."""
    lines = ["<code>", "def func(words: list[str]):"]
    for k in ORDER:
        if k in done:
            lines.append(f"    {ITEMS[k][1]}")
    return "\n".join(lines) + "\n    " + extra   # trailing indent = pivot sits here (about to write next var)

def build_prompt(done, extra=""):
    return get_prompt(QUESTION, build_prefill(done, extra))

def pivot_of(prompt):
    return prepare_input(prompts=prompt, tokenizer=mt)["input_ids"].shape[1] - 1

# task-item token spans are IDENTICAL across done-sets (task precedes the prefill) — compute once.
_p0 = build_prompt([])
ITEM_SPAN = {k: token_span(_p0, ITEMS[k][0]) for k in ORDER}
show_prompt("done={}      ", _p0, ITEM_SPAN["a"])
show_prompt("done={a}     ", build_prompt(["a"]), ITEM_SPAN["b"])
print("item spans:", ITEM_SPAN, " pivot(∅)=", pivot_token(_p0))

## A1 · Graded suppression + OV-write norm

Sweep PROGRESS ∈ {∅, {a}, {a,b}}. For each: value-weighted attention from the pivot onto each task
item's spec tokens (mean over GOAL heads), and the norm of the aggregated goal-head write.
**Predict:** attention onto item-k drops sharply when k enters PROGRESS; frontier item retains mass;
write-norm decreases monotonically toward the DIFF==0 (all-done) floor.

In [ ]:
DONE_LEVELS = [[], ["a"], ["a", "b"]]
tbl = Table(title="A1 · pivot→TASK-item attention (mean over GOAL heads)  +  ‖goal write‖", highlight=False)
tbl.add_column("PROGRESS")
for k in ORDER: tbl.add_column(f"attn·{k}", justify="right")
tbl.add_column("‖write‖", justify="right")
for done in DONE_LEVELS:
    p = build_prompt(done)
    piv = pivot_of(p)
    tok = prepare_input(prompts=p, tokenizer=mt)
    info = get_attention_matrices(tok, mt, value_weighted=True)
    row = ["{" + ",".join(done) + "}"]
    for k in ORDER:
        a = pivot_attn(info, GOAL_HEADS, list(range(*ITEM_SPAN[k])), piv).sum()
        mark = "[green]" if k not in done else "[red]"
        row.append(f"{mark}{a:.3f}[/]")
    row.append(f"{write_norm(p, GOAL_HEADS, piv):.2f}")
    tbl.add_row(*row)
rich_print(tbl)
rich_print("[grey58]green = still-undone item (expect mass), red = done item (expect suppressed)[/grey58]")

## A2 · Anti-alignment of the query update with the completed item's keys

`q` is pre-RoPE at the pivot; `k` is pre-RoPE at item-a's spec tokens. For every GOAL q-head, cosine of
`Δq = q({a}) − q({})` (q-head slice) against the mean item-a key (its kv-head slice, GQA `h//GROUP`).
**Predict:** cosine < 0 (the completed item is *subtracted into* the query, suppressing its keys).

In [ ]:
qheads = heads_per_layer(GOAL_HEADS, "q")
Q_LAYERS = sorted(qheads)
p0, pa = build_prompt([]), build_prompt(["a"])
q0 = cache_src_proj(p0, [pivot_of(p0)], Q_LAYERS, "q")
qa = cache_src_proj(pa, [pivot_of(pa)], Q_LAYERS, "q")
dq = q_lincomb([(1.0, qa), (-1.0, q0)])                       # {layer: (1, width)}

# mean item-a KEY at its spec tokens (positions identical across prompts; use p0)
ka = cache_src_proj(p0, list(range(*ITEM_SPAN["a"])), Q_LAYERS, "k")
kbar = {l: ka[l].mean(0, keepdim=True) for l in Q_LAYERS}     # {layer: (1, width)}

hd = mt.head_dim
coss = []
for (l, h) in GOAL_HEADS:
    if l not in dq: continue
    qv = dq[l][0, h * hd:(h + 1) * hd]
    kvh = h // GROUP
    kv = kbar[l][0, kvh * hd:(kvh + 1) * hd]
    coss.append(cos(qv, kv))
coss = np.array(coss)
print(f"Δq·k(item a) cosine over {len(coss)} GOAL q-heads: "
      f"mean={coss.mean():+.3f}  median={np.median(coss):+.3f}  frac<0={ (coss < 0).mean():.2f}")
print("(strongly negative mean ⇒ completing a subtracts a's key-direction into the query — suppression)")

## A3 · Sign necessity — reflect the done-update

If the *sign* of the query update selects, then reflecting the done-update through baseline —
`q_reflect = q({}) − (q({a,b}) − q({})) = 2·q({}) − q({a,b})` — should **not** behave as if a,b are
done (should not skip to c), unlike the true `q({a,b})`. Injected into the ∅-dest and generated.

In [ ]:
pab = build_prompt(["a", "b"])
qab = cache_src_proj(pab, [pivot_of(pab)], Q_LAYERS, "q")
q_reflect = q_lincomb([(2.0, q0), (-1.0, qab)])

def q_inject_specs(qcache, dest_pivot):
    qh = heads_per_layer(GOAL_HEADS, "q")
    return ([dict(which="q", cache=qcache, heads=qh, positions=[dest_pivot])],
            [dict(which="q", cache=qcache, heads=qh, positions=[-1])])

dest_tok = prepare_input(prompts=p0, tokenizer=mt)
piv0 = pivot_of(p0)
pf_ab, dc_ab = q_inject_specs(qab, piv0)
pf_rf, dc_rf = q_inject_specs(q_reflect, piv0)
rich_print(f"[grey70]clean ∅ (expect writes a) :[/grey70] {escape(repr(greedy(dest_tok, max_new=32)))}")
rich_print(f"[sea_green3]inject q(a,b) (expect c)  :[/sea_green3] "
           f"{escape(repr(greedy(dest_tok, prefill_specs=pf_ab, decode_specs=dc_ab, n_steer=20, max_new=32)))}")
rich_print(f"[salmon1]inject q_reflect (expect ≠c):[/salmon1] "
           f"{escape(repr(greedy(dest_tok, prefill_specs=pf_rf, decode_specs=dc_rf, n_steer=20, max_new=32)))}")

## B1 · Query linearity (geometric)

Is the done-set encoded additively in query space? Cosine, over the concatenated GOAL q-head slices,
between `Δq({a,b})` and `Δq({a}) + Δq({b})`  (each `Δq(·) = q(·) − q({})`).
**Predict:** high cosine ⇒ subtraction is linear/compositional over done-items.

In [ ]:
pb = build_prompt(["b"])
qb = cache_src_proj(pb, [pivot_of(pb)], Q_LAYERS, "q")

def goal_q_vec(qcache):
    parts = [qcache[l][0, h * hd:(h + 1) * hd] for (l, h) in GOAL_HEADS if l in qcache]
    return torch.cat(parts)

dq_a  = goal_q_vec(q_lincomb([(1.0, qa),  (-1.0, q0)]))
dq_b  = goal_q_vec(q_lincomb([(1.0, qb),  (-1.0, q0)]))
dq_ab = goal_q_vec(q_lincomb([(1.0, qab), (-1.0, q0)]))
print(f"cos( Δq(a,b) , Δq(a)+Δq(b) )      = {cos(dq_ab, dq_a + dq_b):+.3f}")
print(f"cos( Δq(a,b) , Δq(a) )            = {cos(dq_ab, dq_a):+.3f}   (single-term baseline)")
print(f"cos( Δq(a)   , Δq(b) )            = {cos(dq_a,  dq_b):+.3f}   (are the two updates ~orthogonal?)")

## B2 · Compositional causal injection  (the payload)

Inject a *synthesised* query `q_synth = q({a}) + q({b}) − q({})` at the ∅-dest pivot and generate.
**Predict:** writes item **c** only — the composed done-set behaves as {a,b}. Controls: real `q({a,b})`
should match; single terms narrow less. **Saturation** `q(a)+q(b)+q(c)−2·q({})` → all done → **boundary/return**.

*Note:* injection target is the fixed-geometry ∅ prompt; only the position-free pre-RoPE queries are combined.

In [ ]:
pc = build_prompt(["c"])
qc = cache_src_proj(pc, [pivot_of(pc)], Q_LAYERS, "q")
q_synth = q_lincomb([(1.0, qa), (1.0, qb), (-1.0, q0)])                       # ≈ done {a,b} -> expect c
q_sat   = q_lincomb([(1.0, qa), (1.0, qb), (1.0, qc), (-2.0, q0)])            # done {a,b,c} -> expect boundary

def gen_inject(qcache, tag, color):
    pf, dc = q_inject_specs(qcache, piv0)
    rich_print(f"[{color}]{tag}:[/{color}] "
               f"{escape(repr(greedy(dest_tok, prefill_specs=pf, decode_specs=dc, n_steer=20, max_new=40)))}")

rich_print(f"[grey70]clean ∅            (expect a) :[/grey70] {escape(repr(greedy(dest_tok, max_new=40)))}")
gen_inject(qab,     "inject q(a,b) real (expect c) ", "sea_green3")
gen_inject(q_synth, "inject q_synth=a+b-∅ (expect c)", "gold3")
gen_inject(qa,      "inject q({a})       (expect b/c)", "grey58")
gen_inject(q_sat,   "inject q_sat=a+b+c-2∅ (→bound)", "salmon1")